# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset with the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', None)} | License: {getattr(meta, 'license', None)}")
print(f"Citation: {getattr(meta, 'citeAs', None)}")
print("\nKeywords:", getattr(meta, 'keywords', None))

## 2. Data Overview
List available record sets and their respective fields/columns using their `@id`s.

In [ ]:
# Show all record sets and their fields

record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    # List all fields and their @id from the record set
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, DataType: {getattr(field, 'dataType', None)})")
    elif hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id}, DataType: {getattr(col, 'dataType', None)})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values listed above.

In [ ]:
# Select record sets by their @id for further analysis
# (If unsure, run the previous cell to print all available record_set @ids)

record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"DataFrame loaded for record set: {record_set_id} | Columns: {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())
        else:
            print(f"No records found for: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Pick a record set and numeric field by their @id for demonstration.
# First, let's print the available record sets and their columns again for reference.
for rs_id, df in dataframes.items():
    print(f"Record set: {rs_id}")
    print(df.columns.tolist())
    print()

# Example: Suppose we select the main protein abundance record set and a column such as 'Abundance_Mean' if it exists.

# For demonstration, we use the first available DataFrame with a likely numeric column
selected_record_set_id = None
numeric_field = None
# Find the first DataFrame with a numeric column
for rs_id, df in dataframes.items():
    for col in df.columns:
        if df[col].dtype in [int, float]:
            selected_record_set_id = rs_id
            numeric_field = col
            break
    if selected_record_set_id:
        break

if selected_record_set_id and numeric_field:
    print(f"Using record set {selected_record_set_id} and numeric field '{numeric_field}' for EDA.")
    # Filtering
    threshold = df[numeric_field].mean() if (df[numeric_field].dtype in [int, float]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered rows where {numeric_field} > {threshold}")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} column:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping - pick another column if available
    group_field = None
    cols = [col for col in df.columns if col != numeric_field]
    for col in cols:
        if df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped mean by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical column for grouping found.")
else:
    print("No numeric field found for EDA demonstration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrates step-by-step loading, exploration, and visualization of the FAIR^2 dataset using the `mlcroissant` library.

- The dataset documents protein abundance and modifications in human samples, providing structured mass spectrometry data.
- Through schema exploration, record set @ids and field @ids allowed selection and processing of desired data tables dynamically.
- Basic EDA highlighted filtering, normalization, and distribution analysis. Grouped analyses are possible if categorical columns are available.

To extend this notebook, try exploring additional record sets or performing domain-relevant analyses and visualizations tailored to research questions.
